# **folium**

In [1]:
import folium
folium.__version__

'0.20.0'

In [2]:
import numpy as np
import pandas as pd
import json

print(f'Numpy:{np.__version__}, Pandas:{pd.__version__}')

Numpy:2.5.2, Pandas:3.0.5


#### 카카오지도 API 사용
- 카카오의 오픈 API
- 하루 접속 건수가 정해져 있어서 사용량을 초과하면 API 사용이 제한될 수 있음

In [3]:
# !pip install geokakao

In [4]:
import geokakao as gk

In [5]:
import displayMap as dm

# 서울시 지하철역 아침 출근 시간대 탑승객 수 지도 시각화

### 1. 지하철 탑승객 데이터셋과 지하철역 주소 데이터 읽기

#### 서울시 지하철 탑승객 데이터
- 데이터 :subway_line_1_8_20231231.csv

In [25]:
df_subway = pd.read_csv('data/subway_line_1_8_20231231.csv')
df_subway.head()

,연번,수송일자,호선명,역번호,역명,승하차구분,승객유형,t06시간대이전,t06_07시간대,t07_08시간대,...,t15_16시간대,t16_17시간대,t17_18시간대,t18_19시간대,t19_20시간대,t20_21시간대,t21_22시간대,t22_23시간대,t23_24시간대,t24시간대이후
0,1,2023-07-01,1,150,서울역,승차,일반,192,192,409,...,2989,3514,3409,2686,2347,1872,2209,1885,691,7
1,2,2023-07-01,1,150,서울역,승차,어린이,0,1,3,...,33,32,30,30,30,16,11,8,4,0
2,3,2023-07-01,1,150,서울역,승차,중고생,0,0,0,...,1,1,0,0,0,0,0,0,0,0
3,4,2023-07-01,1,150,서울역,승차,청소년,0,3,12,...,47,52,54,46,47,47,54,34,13,0
4,5,2023-07-01,1,150,서울역,승차,우대권,88,105,212,...,580,608,560,381,265,277,197,121,31,2


In [7]:
df_subway.info()

<class 'pandas.DataFrame'>
RangeIndex: 817644 entries, 0 to 817643
Data columns (total 27 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   연번         817644 non-null  int64
 1   수송일자       817644 non-null  str  
 2   호선명        817644 non-null  int64
 3   역번호        817644 non-null  int64
 4   역명         817644 non-null  str  
 5   승하차구분      817644 non-null  str  
 6   승객유형       817644 non-null  str  
 7   t06시간대이전   817644 non-null  int64
 8   t06_07시간대  817644 non-null  int64
 9   t07_08시간대  817644 non-null  int64
 10  t08_09시간대  817644 non-null  int64
 11  t09_10시간대  817644 non-null  int64
 12  t10_11시간대  817644 non-null  int64
 13  t11_12시간대  817644 non-null  int64
 14  t12_13시간대  817644 non-null  int64
 15  t13_14시간대  817644 non-null  int64
 16  t14_15시간대  817644 non-null  int64
 17  t15_16시간대  817644 non-null  int64
 18  t16_17시간대  817644 non-null  int64
 19  t17_18시간대  817644 non-null  int64
 20  t18_19시간대  817644 non-null  int64
 21

In [8]:
# c.columns

In [9]:
df_subway.수송일자

0         2023-07-01
1         2023-07-01
2         2023-07-01
3         2023-07-01
4         2023-07-01
             ...    
817639    2023-12-31
817640    2023-12-31
817641    2023-12-31
817642    2023-12-31
817643    2023-12-31
Name: 수송일자, Length: 817644, dtype: str

In [11]:
df_subway_202607= pd.read_csv('data/CARD_SUBWAY_MONTH_202607.csv')
df_subway_202607.head()

,사용일자,노선명,역명,승차총승객수,하차총승객수,등록일자
20260701,경원선,외대앞,13163,12934,20260704,NaN
20260701,6호선,역촌,4368,5180,20260704,NaN
20260701,6호선,응암,19333,19549,20260704,NaN
20260701,5호선,하남검단산,7496,6946,20260704,NaN
20260701,5호선,하남시청(덕풍·신장),9957,9404,20260704,NaN


#### 서울시 지하철역 주소 
- 데이터 : seoul_subway_address_2023,csv

In [12]:
df_addr = pd.read_csv('data/seoul_subway_address_2023.csv')
df_addr.head()

,연번,역번호,호선,역명,역전화번호,도로명주소,지번주소
0,1,150,1,서울,02-6110-1331,서울특별시 중구 세종대로 지하2(남대문로 5가),서울특별시 중구 남대문로5가 73-6 서울역(1호선)
1,2,151,1,시청,02-6110-1321,서울특별시 중구 세종대로 지하101(정동),서울특별시 중구 정동 5-5 시청역(1호선)
2,3,152,1,종각,02-6110-1311,서울특별시 종로구 종로 지하55(종로1가),서울특별시 종로구 종로1가 54 종각역(1호선)
3,4,153,1,종로3가,02-6110-1301,서울특별시 종로구 종로 지하129(종로3가),서울특별시 종로구 종로3가 10-5 종로3가역(1호선)
4,5,154,1,종로5가,02-6110-1291,서울특별시 종로구 종로 지하216(종로5가),서울특별시 종로구 종로5가 82-1 종로5가역(1호선)


In [13]:
df_addr.info()

<class 'pandas.DataFrame'>
RangeIndex: 288 entries, 0 to 287
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   연번      288 non-null    int64
 1   역번호     288 non-null    int64
 2   호선      288 non-null    int64
 3   역명      288 non-null    str  
 4   역전화번호   288 non-null    str  
 5   도로명주소   288 non-null    str  
 6   지번주소    288 non-null    str  
dtypes: int64(3), str(4)
memory usage: 15.9 KB


### 2. 2호선 오전 8~9시대 탑승객 수를 역별로 집계하기

In [27]:
time_zone = ['t07_08시간대', 't08_09시간대']
df_subway_line2 = df_subway.loc[df_subway.호선명 == 2]  # 2호선 추출
df_subway_line2.value_counts()

연번      수송일자        호선명  역번호  역명         승하차구분  승객유형     t06시간대이전  t06_07시간대  t07_08시간대  t08_09시간대  t09_10시간대  t10_11시간대  t11_12시간대  t12_13시간대  t13_14시간대  t14_15시간대  t15_16시간대  t16_17시간대  t17_18시간대  t18_19시간대  t19_20시간대  t20_21시간대  t21_22시간대  t22_23시간대  t23_24시간대  t24시간대이후
205     2023-07-01  2    201  시청         승차     일반       21        38         86         197        252        400        581        719        886        1113       1342       1651       1758       2031       1248       970        882        602        167        0           1
206     2023-07-01  2    201  시청         승차     어린이      0         0          0          1          5          4          12         8          20         30         27         38         30         30         23         10         13         13         0          0           1
207     2023-07-01  2    201  시청         승차     중고생      0         0          0          0          0          0          1          0          0          0          0 

In [28]:
df_subway_line2.승하차구분.value_counts()

승하차구분
승차    84942
하차    84942
Name: count, dtype: int64

In [30]:
#                                time_zone = ['t07_08시간대', 't08_09시간대']
df_subway_line2 = df_subway.groupby('역명')[time_zone].sum()  # 역별 승객 수 집계
df_subway_line2['탑승객 수'] = df_subway.sum(axis=1)  # 탑승객 수 합계
df_subway_line2.head()

,t07_08시간대,t08_09시간대,탑승객 수
역명,,,
가락시장,354770,599691,1908922
가산디지털단지,949026,2415882,6729816
강남,1113322,2136584,6499812
강남구청,254166,672293,1852918
강동,494846,700387,2390466


### 3. 탑승객 데이터와 역 주소 데이터를 병합한 후, 주소를 이용해 위도와 경도 값을 추가하기

In [31]:
df_merge = pd.merge(df_subway_line2, df_addr, on='역명', how='inner')

# add lat, lon
gk.add_coordinates_to_dataframe(df_merge, '도로명주소')

# 문자열 좌푯값을 숫자로 변환
df_merge.decimalLatitude = pd.to_numeric(df_merge.decimalLatitude)
df_merge.decimalLongitude = pd.to_numeric(df_merge.decimalLongitude)
df_merge.info()

<class 'pandas.DataFrame'>
RangeIndex: 268 entries, 0 to 267
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   역명                268 non-null    str    
 1   t07_08시간대         268 non-null    int64  
 2   t08_09시간대         268 non-null    int64  
 3   탑승객 수             268 non-null    int64  
 4   연번                268 non-null    int64  
 5   역번호               268 non-null    int64  
 6   호선                268 non-null    int64  
 7   역전화번호             268 non-null    str    
 8   도로명주소             268 non-null    str    
 9   지번주소              268 non-null    str    
 10  decimalLatitude   268 non-null    float64
 11  decimalLongitude  268 non-null    float64
dtypes: float64(2), int64(6), str(4)
memory usage: 25.3 KB


### 4. 지도의 중심점을 계산하기

In [16]:
center = df_merge[['decimalLatitude', 'decimalLongitude']].mean().to_list()

In [17]:
center

[37.53262927813688, 126.99181101601665]

### 5. 역의 위치를 지도 위에 마커로 표시하기

In [18]:
# 지도 객체 생성
map = folium.Map(location=center, zoom_start=12)

In [19]:
# 지도에 마커 추가
for i in range(len(df_merge)):
    folium.Marker(location=[df_merge.loc[i, 'decimalLatitude'],
                            df_merge.loc[i, 'decimalLongitude']],
                            icon=folium.Icon(color='red', icon='star')).add_to(map)
dm.showMap(map)

### 6. 역별 탑승객 수를 역의 위치에 원으로 표시하고, 역 이름도 함께 나타내기

In [20]:
# 지도 객체 생성
map = folium.Map(location=center, zoom_start=12)

for i in range(len(df_merge)):
    folium.CircleMarker(location=[df_merge.loc[i, 'decimalLatitude'],
    df_merge.loc[i, 'decimalLongitude']],
    radius=((df_merge.loc[i, '탑승객 수']/150000)),  # 원의 반지름
    color='red',  # 원의 색
    stroke=False,  # 윤곽선 없음
    fill=True,  # 원의 내부 색
    fill_opacity='50%'  # 원의 내부 색 투명도
    ).add_to(map)

In [21]:
html_start = html = '<div \
style="\
font-size: 12px;\
color: blue;\
background-color:rgba(255, 255, 255, 0.2);\
width:85px;\
text-align:left;\
margin:0px;\
"><b>'
html_end = '</b></div>'

for i in range(len(df_merge)):
    folium.Marker(location=[df_merge.loc[i, 'decimalLatitude'],
    df_merge.loc[i, 'decimalLongitude']],
    icon=folium.DivIcon(
        icon_anchor=(0, 0),  # 텍스트 위치 설정
        html=html_start+df_merge.loc[i, '역명']+html_end
        )).add_to(map)

# 웹 브라우저에 지도 출력
dm.showMap(map)

------